In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import os
from pathlib import Path
import statsmodels.formula.api as smf
import pyarrow.parquet as pq

In [2]:
ROOT = Path("/home/hanwenying")
DATADIR = ROOT / "rothman-data/transcriptlength"

In [3]:
tpm = pq.ParquetFile(DATADIR / "tpm-gtex-v11.parquet")

In [4]:
tpm.metadata

  created_by: parquet-cpp-arrow version 21.0.0
  num_columns: 19790
  num_rows: 74628
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 11730435

In [ ]:
genes = pq.read_table(DATADIR / "tpm-gtex-v11.parquet", columns=['Name']).to_pandas().reset_index()
genes = genes.squeeze()

In [17]:
metadata = pd.read_csv(DATADIR / "metadata_dummy.tsv", sep='\t', dtype=str)

In [13]:
gene_tstats = pd.DataFrame(columns=['GENE', 'TSTATISTIC', 'PVALUE'])

In [5]:
# agemap = {'20-29':0, '30-39':1, '40-49':2, '50-59':3, '60-69':4, '70-79':5}
# metadata['AGE'] = metadata['AGE'].map(agemap)
# metadata['SEX'] = metadata['SEX'].astype(int) - 1

# metadata.to_csv(DATADIR / 'metadata_dummy.tsv', sep='\t', index=False)

In [6]:
gene_tpm = next(tpm.iter_batches(batch_size=1)).to_pandas().T

In [7]:
gene_tpm = gene_tpm.reset_index()
gene_tpm = gene_tpm.rename_axis(None, axis=1)
ensembl = gene_tpm.columns[1]
gene_tpm.columns = ['SAMPLE', 'TPM']
gene_tpm = gene_tpm[gene_tpm['SAMPLE'] != 'Description']

In [8]:
dm = pd.merge(gene_tpm, metadata, on='SAMPLE')

In [9]:
dm['TPM'] = dm['TPM'].astype(float)
dm['AGE'] = dm['AGE'].astype(int)

In [10]:
model = smf.mixedlm("TPM ~ AGE + SEX + TISSUE", dm, groups=dm['SUBJID'])

In [11]:
res = model.fit()

/home/hanwenying/.conda/envs/bio/lib/python3.14/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [12]:
t, p = res.tvalues['AGE'], res.pvalues['AGE']

In [14]:
gene_tstats.loc[len(gene_tstats)] = [ensembl, t, p]